Ran on March 17th 5:00

In [1]:
import pandas as pd
import os
import cv2
import numpy as np
import json

In [2]:
def Detect_Office(Json,Office):

    NewList=Json['fields']
    Dict=list()
    for d in NewList:
        try:
            newDict={}
            newDict['inferText']=d['inferText']
            newDict['boundingPoly']=d['boundingPoly']
            Dict.append(newDict)
        except KeyError:
            continue

    res = [d
       for d in Dict 
       if (Office[0] == d['inferText'][0]) or (Office[0] == d['inferText']) or (Office == d['inferText']) or (Office[-2:][0] == d['inferText'])]

    if len(res)!=0:
        res = res[0]['boundingPoly']['vertices']
        Edge=max(int(d['x']) for d in res)
        return(Edge)
    else:
        return(None)

def Detect_Office2(Json,Office):
    NewList=Json['fields']
    Dict=list()
    for d in NewList:
        try:
            newDict={}
            newDict['inferText']=d['inferText']
            newDict['boundingPoly']=d['boundingPoly']
            Dict.append(newDict)
        except KeyError:
            continue

    res = [d
       for d in Dict 
       if (Office[0] == d['inferText']) or (Office == d['inferText']) or (Office[-2:][0] == d['inferText'])]

    if len(res)!=0:
        res = res[0]['boundingPoly']['vertices']
        Edge=max(int(d['x']) for d in res)
        return(Edge)
    else:
        return(None)

class NpEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)

In [3]:
### CLOVA FUNCTION ###
import requests
import uuid
import time
import json
import cv2
import base64

api_url = 'https://deelieyxuc.apigw.ntruss.com/custom/v1/1972/ebd01bcbf693d069817622e9839e20914143c7d0d8953eddee40e8b0af96c95b/general'
secret_key = 'S1NmVXpYZlJ0cGJ0ZEFRZXVlbkRkaHFReE9FcHNTQ0U='

def Clova(Year,Page):
    path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
    with open(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg",'rb') as f:
         file_data = f.read()

    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(file_data).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)['images'][0]
    
    return Json

def Clova2(Year,Page,Top):
    path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
    stream = open(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg", "rb")
    bytes = bytearray(stream.read())
    numpyarray = np.asarray(bytes, dtype=np.uint8)
    img = cv2.imdecode(numpyarray, cv2.IMREAD_UNCHANGED)
    HH,WW=img.shape[:2]
    cropped=img[0:Top,0:WW]
    temp_path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Temp\\"
    cv2.imwrite(temp_path+"Temp.jpg",cropped)
    
    with open(temp_path+"Temp.jpg",'rb') as f:
        file_data = f.read()

    request_json = {
            'images': [
                {
                    'format': 'jpg',
                    'name': 'demo',
                    'data':base64.b64encode(file_data).decode()}],
            'requestId': str(uuid.uuid4()),
            'version': 'V2',
            'timestamp': int(round(time.time() * 1000)),
            'lang':'ja'
            }
    payload = json.dumps(request_json).encode("UTF-8")
    headers = {'X-OCR-SECRET': secret_key,
              'Content-Type': 'application/json'}
    response = requests.request("POST", api_url, headers=headers, data = payload)
    Json=json.loads(response.text)['images'][0]
    
    return Json

In [26]:
def Show(dta,Dept,Office):
    Page=int(dta[Year][Dept][Office]['Starting_Page'])
    StrLoc=int(dta[Year][Dept][Office]['Office_X1'])
    img=cv2.imread("Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
    HH,WW=img.shape[:2]
    Annotate=img.copy()
    Annotate=cv2.line(img,(StrLoc,0),(StrLoc,HH),(0,0,0),1)
    cv2.imshow(Dept+Office,Annotate)
    cv2.waitKey(0)

In [27]:
Year='1935'
Showa='10'
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
os.chdir(path)
df = pd.read_csv(r'C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs/Processed_Data/Index/S'+Showa+'.csv')
df=df.drop(df.columns[0], axis=1)

file_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'+Year+'\\DataFrame.json'
with open(file_path, encoding="utf-8") as f:
    dta = json.loads(f.read())

In [28]:
n=0
Row  = df.iloc[n]
Page=int(Row["Page"])
Office=Row["Office"]
Dept=Row['Dept']
print(Office,Dept)
dta[Year][Dept][Office]={}
dta[Year][Dept][Office]["Starting_Page"]=Page

img=cv2.imread("Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
#Convert to json via CLOVA
Json=Clova(Year,Page)

#Find X coordinate of 'Office'.
XCoord_Unit=Detect_Office(Json,Office)
dta[str(Year)][Dept][Office]["Office_X1"]=XCoord_Unit
HH=img.shape[:2][0]
print(Office+ 'success')
cv2.line(img, (XCoord_Unit,0), (XCoord_Unit,HH), (255,0,0), 2)
cv2.imshow('pic',img)
cv2.waitKey(0)

秘書課 Admin
秘書課success


0

In [6]:
#Test code| Version 2#
#Show Working office#
FailedList=[]
for n in range(1,len(df)):
    #Extract key info of office
    Row  = df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    ###Insert Starting page information to motherframe###
    dta[Year][Dept][Office]={}
    dta[Year][Dept][Office]["Starting_Page"]=Page
    print('['+str(n)+',"'+Dept+'","'+Office+'"]')

    ###Collect Location information###
    ##Read image for first page##
    img=cv2.imread("Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
    #Convert to json via CLOVA
    Json=Clova(Year,Page)

    #Find X coordinate of 'Office'.
    XCoord_Unit=Detect_Office(Json,Office)
    if XCoord_Unit==None:
        #Add to motherframe
        dta[str(Year)][Dept][Office]["Office_X1"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Ending_Page"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Office_X2"]='NA'
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]='NA'
        print(Office+ 'failed')
        FailedList.append(list((n,Dept,Office)))
        continue
    else:
        dta[str(Year)][Dept][Office]["Office_X1"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Ending_Page"]=Page
        dta[str(Year)][ExDept][ExOffice]["Office_X2"]=XCoord_Unit+10
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        HH=img.shape[:2][0]
        print(Office+ 'success')
        cv2.line(img, (XCoord_Unit,0), (XCoord_Unit,HH), (255,0,0), 2)
        cv2.imshow('pic',img)
        cv2.waitKey(0)

[1,"Admin","職員課"]
職員課success
[2,"Admin","文書課"]
文書課success
[3,"監査局","庶務課"]
庶務課success
[4,"監査局","監察課"]
監察課success
[5,"監査局","区政課"]
区政課success
[6,"監査局","統計課"]
統計課success
[7,"監査局","都市計画課"]
都市計画課success
[8,"財務局","主計課"]
主計課success
[9,"財務局","公債課"]
公債課success
[10,"財務局","収納課"]
収納課success
[11,"財務局","経理課"]
経理課success
[12,"財務局","地理課"]
地理課success
[13,"財務局","会計課"]
会計課success
[14,"産業局","庶務課"]
庶務課success
[15,"産業局","商工課"]
商工課success
[16,"産業局","農魚課"]
農魚課success
[17,"産業局","権度課"]
権度課success
[18,"教育局","庶務課"]
庶務課success
[19,"教育局","学務課"]
学務課failed
[20,"教育局","社会教育課"]
社会教育課success
[21,"教育局","体育課"]
体育課success
[22,"教育局","視学課"]
視学課success
[23,"社会局","庶務課"]
庶務課success
[24,"社会局","保護課"]
保護課success
[25,"社会局","福利課"]
福利課success
[26,"社会局","職業課"]
職業課success
[27,"保健局","庶務課"]
庶務課success
[28,"保健局","衛生課"]
衛生課success
[29,"保健局","清掃課"]
清掃課success
[30,"保健局","公園課"]
公園課success
[31,"水道局","庶務課"]
庶務課success
[32,"水道局","会計課"]
会計課success
[33,"水道局","業務課"]
業務課success
[34,"水道局","給水課"]
給水課success
[35,"水道局","拡張課"]
拡張課success
[36,"土木局","庶務課"]
庶

In [7]:
Type1=[[22,"教育局","視学課"],
      [26,"社会局","職業課"],
      [36,"土木局","庶務課"],
      [56,"中央卸売市場","企画課"],
      [57,"養育院","院長室"]]

In [8]:
FailRate=len(FailedList)/len(df)
if len(FailedList)/len(df)<0.2:
    print('Fantastic!! Success Rate is '+str(1-(len(FailedList)/len(df))))
else:
    print('残念...Success Rate is '+str(1-(len(FailedList)/len(df))))
DF=pd.read_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv')
DF.loc[int(Year)-1934, 'Office'] = 1-FailRate
DF.to_csv('C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\Records.csv')

Fantastic!! Success Rate is 0.984375


In [20]:
#Fixing Failed Offices
#Step1: Check for simple page assignment error
path="C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Raw_Data\\Splited\\"+Year+"\\"
for n in range(0,len(FailedList)):
    Dept=FailedList[n][1]
    Office=FailedList[n][2]
    print(Dept,Office)
    Page=df['Page'][(df['Office']==Office) & (df['Dept']==Dept)].tolist()[0]
    image=cv2.imread(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
    cv2.imshow('Image',image)
    cv2.waitKey(0)

教育局 学務課


PageSplit Error

電気局 人事掛:p116,p117
電気局 庶務課:p116,p117

=> Fixed

In [21]:
FailedList2=[]
for Office in FailedList:
    n=Office[0]
    Row=df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    Json=Clova2(Year,Page,150)
    
    try:
        XCoord_Unit=Detect_Office2(Json,Office)
        dta[str(Year)][Dept][Office]["Office_X1"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Ending_Page"]=Page
        dta[str(Year)][ExDept][ExOffice]["Office_X2"]=XCoord_Unit+10
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row '+str(n))
        img=cv2.imread(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
        cv2.line(img, (XCoord_Unit,0), (XCoord_Unit,300), (255,0,0), 2)
        cv2.imshow('pic',img)
        cv2.waitKey(0)
    except:
        FailedList2.append(list((n,Dept,Office)))
        continue

学務課success: at row 19


In [22]:
FailedList2

[]

In [25]:
Type1_2=[]
Top=120
for Office in Type1:
    n=Office[0]
    Row=df.iloc[n]
    ExRow= df.iloc[n-1]

    Page=int(Row["Page"])
    Office=Row["Office"]
    Dept=Row['Dept']

    ExPage=int(ExRow["Page"])
    ExOffice=ExRow["Office"]
    ExDept=ExRow['Dept']

    Json=Clova2(Year,Page,Top)
    
    try:
        XCoord_Unit=Detect_Office2(Json,Office)
        dta[str(Year)][Dept][Office]["Office_X1"]=XCoord_Unit
        dta[str(Year)][ExDept][ExOffice]["Ending_Page"]=Page
        dta[str(Year)][ExDept][ExOffice]["Office_X2"]=XCoord_Unit+10
        dta[str(Year)][ExDept][ExOffice]["Page_Range"]=list(range(ExPage,Page+1))       
        print(Office+ 'success: at row '+str(n))
        img=cv2.imread(path+"Page"+"{:03d}".format(Page)+"\\"+"Page"+"{:03d}".format(Page)+".jpg")
        cv2.line(img, (XCoord_Unit,0), (XCoord_Unit,Top), (255,0,0), 2)
        cv2.imshow('pic',img)
        cv2.waitKey(0)
    except:
        Type1_2.append(list((n,Dept,Office)))
        continue

視学課success: at row 22
職業課success: at row 26
庶務課success: at row 36
企画課success: at row 56
院長室success: at row 57


In [28]:
json_object = json.dumps(dta, indent=4,
                        cls=NpEncoder)
save_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'+str(Year)+'\\'
with open(save_path+"DataFrame_Office.json", "w") as outfile:
    outfile.write(json_object)

In [15]:
file_path='C:\\Users\\Keitaro Ninomiya\\Box\\Research Notes (keitaro2@illinois.edu)\\Tokyo_Jobs\\Processed_Data\\'+Year+'\\DataFrame_Office.json'
with open(file_path, encoding="utf-8") as f:
    dta = json.loads(f.read())

In [5]:
Clova('1935',4)

{'uid': 'b9aea91c5eb4499ba96095f556754aa4',
 'name': 'demo',
 'inferResult': 'SUCCESS',
 'message': 'SUCCESS',
 'validationResult': {'result': 'NO_REQUESTED'},
 'convertedImageInfo': {'width': 582,
  'height': 420,
  'pageIndex': 0,
  'longImage': False},
 'fields': [{'valueType': 'ALL',
   'boundingPoly': {'vertices': [{'x': 483.0, 'y': 62.0},
     {'x': 503.0, 'y': 62.0},
     {'x': 503.0, 'y': 83.0},
     {'x': 483.0, 'y': 83.0}]},
   'inferText': '文',
   'inferConfidence': 1.0,
   'type': 'NORMAL',
   'lineBreak': True},
  {'valueType': 'ALL',
   'boundingPoly': {'vertices': [{'x': 52.0, 'y': 60.0},
     {'x': 61.0, 'y': 60.0},
     {'x': 61.0, 'y': 76.0},
     {'x': 52.0, 'y': 76.0}]},
   'inferText': '入上',
   'inferConfidence': 0.9852,
   'type': 'NORMAL',
   'lineBreak': False},
  {'valueType': 'ALL',
   'boundingPoly': {'vertices': [{'x': 69.0, 'y': 59.0},
     {'x': 77.0, 'y': 60.0},
     {'x': 76.0, 'y': 76.0},
     {'x': 68.0, 'y': 75.0}]},
   'inferText': '七下',
   'inferCon